To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

In [3]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm
# Install latest Hugging Face for Gemma-3!
!pip install --no-deps git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3

In [4]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

dataset_path = '/content/drive/MyDrive/simplification_final_dataset.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from unsloth import FastModel
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

==((====))==  Unsloth 2025.3.19: Fast Gemma3 patching. Transformers: 4.50.0.dev0. vLLM: 0.8.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/4.56G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

We now add LoRA adapters so we only need to update a small amount of parameters!

In [17]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.language_model.model` require gradients


In [5]:
# ✅ Load Dataset
from datasets import load_dataset
dataset = load_dataset("csv", data_files=dataset_path)['train']

# ✅ Format dataset using Chat Template
def format_chat(example):
    return {
        "messages": [
            {"role": "user", "content": f"You are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\n{example['legal_text']}"},
            {"role": "assistant", "content": f'Simplified Explanation: {example["simplified_text"]}'},
        ]
    }

dataset = dataset.map(format_chat)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2126 [00:00<?, ? examples/s]

In [8]:
dataset[1]['messages']



[{'content': 'You are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\nOn the appointed day―\n(a) the Common and all other property which immediately before that\ndate was the property of the Churchwardens and was used or held in\nconnection with the Common; and\n(b) all rights and liabilities of the Churchwardens subsisting immediately\nbefore that date which were acquired or incurred in connection with\nthe Common, are transferred to and vest in the Trust free of any trusts established under the 1777 Act.',
  'role': 'user'},
 {'content': 'Simplified Explanation: Transfer of Property and Rights: On the specified date:\nThe property and all other assets that previously belonged to the Chur

In [11]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

We now use `standardize_data_formats` to try converting datasets to the correct format for finetuning purposes!

In [12]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

Let's see how row 100 looks like!

In [13]:
dataset[100]

{'legal_text': 'Whilst the Common is in its ownership―\n(a) the Trust must remain a charity;\n(b) the objects of the Trust must include the primary objects.\nFrom the appointed day the Churchwardens shall not be liable for any act, event, failure to act or omission so far as the act, event, failure to act or omission relates to the Common and occurred before the appointed day.\nWhere the transfer and vesting of the Common or any part of the Common effected by subsection (1) is a registrable disposition under the Land Registration Act 2002, the Trust must apply to the Chief Land Registrar for registration in the register of title of a restriction to reflect section 12(2).',
 'simplified_text': "While the Common is owned by the Trust:\nThe Trust must continue to operate as a charity.\nThe Trust's mission must include the main objectives.\n\nFrom the appointed day, the Churchwardens will not be responsible for any actions, events, failures to act, or omissions related to the Common that o

We now have to apply the chat template for `Gemma-3` onto the conversations, and save it to `text`

In [15]:
def apply_chat_template(examples):
    texts = tokenizer.apply_chat_template(examples["messages"])
    return { "text" : texts }
pass
dataset = dataset.map(apply_chat_template, batched = True)

Map:   0%|          | 0/2126 [00:00<?, ? examples/s]

Let's see how the chat template did! Notice `Gemma-3` default adds a `<bos>`!

In [16]:
dataset[100]["text"]

"<bos><start_of_turn>user\nYou are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\nWhilst the Common is in its ownership―\n(a) the Trust must remain a charity;\n(b) the objects of the Trust must include the primary objects.\nFrom the appointed day the Churchwardens shall not be liable for any act, event, failure to act or omission so far as the act, event, failure to act or omission relates to the Common and occurred before the appointed day.\nWhere the transfer and vesting of the Common or any part of the Common effected by subsection (1) is a registrable disposition under the Land Registration Act 2002, the Trust must apply to the Chief Land Registrar for registration in the register of 

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [18]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2126 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [19]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=2):   0%|          | 0/2126 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again:

In [20]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

"<bos><bos><start_of_turn>user\nYou are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\nWhilst the Common is in its ownership―\n(a) the Trust must remain a charity;\n(b) the objects of the Trust must include the primary objects.\nFrom the appointed day the Churchwardens shall not be liable for any act, event, failure to act or omission so far as the act, event, failure to act or omission relates to the Common and occurred before the appointed day.\nWhere the transfer and vesting of the Common or any part of the Common effected by subsection (1) is a registrable disposition under the Land Registration Act 2002, the Trust must apply to the Chief Land Registrar for registration in the registe

Now let's print the masked out example - you should see only the answer is present:

In [21]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

"                                                                                                                                                                                                                           Simplified Explanation: While the Common is owned by the Trust:\nThe Trust must continue to operate as a charity.\nThe Trust's mission must include the main objectives.\n\nFrom the appointed day, the Churchwardens will not be responsible for any actions, events, failures to act, or omissions related to the Common that occurred before the appointed day.\n\nIf the transfer of the Common or any part of it is a registrable transaction under the Land Registration Act 2002, the Trust must request the Chief Land Registrar to register a restriction in the title register to reflect section 12(2).<end_of_turn>\n"

In [36]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
14.275 GB of memory reserved.


In [23]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,126 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,901,248/4,000,000,000 (0.37% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.219800
2,3.280800
3,3.278000
4,3.298900
5,2.391900
6,2.181900
7,1.409100
8,1.055100
9,0.948300
10,0.870700


In [30]:
model.save_pretrained("/content/drive/MyDrive/legal_nlp/gemma_legal_simplifier")
tokenizer.save_pretrained("/content/drive/MyDrive/legal_nlp/gemma_legal_simplifier")

['/content/drive/MyDrive/legal_nlp/gemma_legal_simplifier/processor_config.json']

In [42]:
import gc
tensor = None
gc.collect()
torch.cuda.empty_cache()


In [43]:
print(torch.cuda.memory_summary(device=None, abbreviated=False))

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   4506 MiB |  14586 MiB |  12320 GiB |  12316 GiB |
|       from large pool |   4376 MiB |  14368 MiB |  12079 GiB |  12075 GiB |
|       from small pool |    130 MiB |    285 MiB |    241 GiB |    241 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   4506 MiB |  14586 MiB |  12320 GiB |  12316 GiB |
|       from large pool |   4376 MiB |  14368 MiB |  12079 GiB |

In [ ]:

from unsloth import FastLanguageModel
import evaluate
from transformers import pipeline


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/legal_nlp/gemma_legal_simplifier",  # Path to your fine-tuned model
    dtype = torch.float16,
    load_in_4bit = True,
    use_safetensors = True,
    device_map = "auto"
)

# ✅ Enable inference optimizations
FastLanguageModel.for_inference(model)

# ✅ Load a small test subset
test_dataset = load_dataset("csv", data_files=dataset_path)["train"].select(range(20))

# ✅ Create generation prompts using chat template
def make_chat_prompt(legal_text):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": f"You are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\n{example['legal_text']}"}],
        tokenize=False,
        add_generation_prompt=True
    )

predictions, references = [], []

for example in test_dataset:
    prompt = make_chat_prompt(example["legal_text"])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove the prompt portion to get only the response
    simplified_output = decoded.split("Simplify the following legal text:")[-1].strip().split("\n")[-1]
    predictions.append(simplified_output.strip())
    references.append(example["simplified_text"].strip())

# ✅ Evaluate with ROUGE, BLEU, BERTScore
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
bertscore = evaluate.load("bertscore")

print("🔍 ROUGE:", rouge.compute(predictions=predictions, references=references))
print("🔍 BLEU:", bleu.compute(predictions=[p.split() for p in predictions], references=[[r.split()] for r in references]))
print("🔍 BERTScore:", bertscore.compute(predictions=predictions, references=references, lang="en"))


==((====))==  Unsloth 2025.3.19: Fast Gemma3 patching. Transformers: 4.50.0.dev0. vLLM: 0.8.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1068.4322 seconds used for training.
17.81 minutes used for training.
Peak reserved memory = 13.561 GB.
Peak reserved memory for training = 9.278 GB.
Peak reserved memory % of max memory = 91.995 %.
Peak reserved memory for training % of max memory = 62.94 %.
